<a href="https://colab.research.google.com/github/Dhanush-Kalaparthi/Downtime_Reduction_Dashboard_2026/blob/main/Updated_RAM_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd

os.makedirs('config', exist_ok=True)

# Dataset with Weibull parameters and parallel redundancy logic
data = """id,name,stage,beta,eta,mttr_mean,mttr_sigma,parallel_units,nominal_tph,crew_required
CR01,Primary Crusher,Crushing,2.5,2200,12,2.5,1,1250,2
CV01,Main Feed Conveyor,Conveying,1.8,3500,4,1.0,1,1300,1
SAG1,SAG Mill 01,Grinding,2.0,1800,48,8.0,2,850,3
SAG2,SAG Mill 02,Grinding,2.0,1800,48,8.0,2,850,3
BM01,Ball Mill 01,Grinding,2.2,2000,36,6.0,2,650,2
BM02,Ball Mill 02,Grinding,2.2,2000,36,6.0,2,650,2
PMP1,Slurry Pump Alpha,Pumping,1.5,800,6,1.5,3,1100,1
FLOT,Flotation Bank,Processing,2.4,4000,24,4.0,1,1800,2
THK1,Thickener Unit,Tailings,3.0,6000,72,12.0,1,2000,2"""

with open("config/equipment_data.csv", "w") as f:
    f.write(data)
print("✅ Industry Dataset Created.")

✅ Industry Dataset Created.


In [ ]:
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from collections import defaultdict, deque

# --- 1. THE ARCHITECTURE ---

@dataclass(order=True)
class Event:
    time: float
    etype: str = field(compare=False)
    eq_id: str = field(compare=False)
    priority: int = field(default=2, compare=True) # 1: RepairEnd, 2: Failure/PM

class Equipment:
    def __init__(self, **kwargs):
        for k, v in kwargs.items(): setattr(self, k, v)
        self.state = "working"
        self.total_downtime = 0.0
        self.total_repairs = 0.0
        self.failures = 0
        self.last_event_time = 0.0
        self.repair_time_history = []
        self.ttf_history = []

    def sample_ttf(self):
        return np.random.weibull(float(self.beta)) * float(self.eta)

    def sample_repair_time(self, ldt):
        base_repair = np.random.lognormal(np.log(float(self.mttr_mean)), float(self.mttr_sigma))
        return base_repair + ldt # Maintainability = Wrench Time + Logistical Delay

class RAM_V3_Engine:
    def __init__(self, df, horizon, crews, ldt, strategy, pm_int, seed):
        self.horizon, self.crews, self.ldt = horizon, crews, ldt
        self.strategy, self.pm_int = strategy, pm_int
        self.current_time, self.available_crews = 0.0, crews
        self.event_queue, self.repair_queue = [], deque()
        self.history = [] # (time, throughput)

        np.random.seed(seed)
        self.equipments = {row['id']: Equipment(**row.to_dict()) for _, row in df.iterrows()}
        self.stages = defaultdict(list)
        for eq in self.equipments.values(): self.stages[eq.stage].append(eq)
        self._init_events()

    def _init_events(self):
        for eq in self.equipments.values():
            ttf = eq.sample_ttf()
            heapq.heappush(self.event_queue, Event(ttf, 'failure', eq.id, 2))
            if self.strategy == 'preventive':
                heapq.heappush(self.event_queue, Event(self.pm_int, 'pm', eq.id, 2))

    def _process_repair_queue(self):
        while self.repair_queue:
            eq = self.equipments[self.repair_queue[0]]
            if self.available_crews >= eq.crew_required:
                self.repair_queue.popleft()
                self.available_crews -= eq.crew_required
                repair_t = eq.sample_repair_time(self.ldt)
                eq.repair_time_history.append(repair_t)
                heapq.heappush(self.event_queue, Event(self.current_time + repair_t, 'repair_end', eq.id, 1))
            else: break

    def run(self):
        while self.event_queue and self.current_time < self.horizon:
            ev = heapq.heappop(self.event_queue)
            self.current_time = ev.time
            eq = self.equipments[ev.eq_id]

            if ev.etype in ['failure', 'pm']:
                if eq.state == "working":
                    eq.state, eq.failures, eq.last_event_time = "failed", eq.failures + 1, self.current_time
                    self.repair_queue.append(eq.id)
                    self._process_repair_queue()
            elif ev.etype == 'repair_end':
                eq.total_downtime += (self.current_time - eq.last_event_time)
                eq.state, self.available_crews = "working", self.available_crews + eq.crew_required
                heapq.heappush(self.event_queue, Event(self.current_time + eq.sample_ttf(), 'failure', eq.id, 2))
                self._process_repair_queue()

            self.history.append((self.current_time, self._get_tph()))

        return self._summarize()

    def _get_tph(self):
        rates = [sum(e.nominal_tph for e in eqs if e.state == "working") for eqs in self.stages.values()]
        return min(rates) if rates else 0

    def _summarize(self):
        avails = [1 - (e.total_downtime / self.horizon) for e in self.equipments.values()]
        obs_mttr = np.mean([np.mean(e.repair_time_history) if e.repair_time_history else 0 for e in self.equipments.values()])
        return np.mean(avails), np.mean([h[1] for h in self.history]), obs_mttr

# --- 3. INTERACTIVE DASHBOARD WRAPPER ---

def run_v3_interface():
    print("═══ 💎 INDUSTRY RAM-DES V3.0: PROFESSIONAL DECISION TOOL ═══")
    try:
        # User Inputs
        target_a = float(input("🎯 Target System Availability (e.g. 0.85): "))
        horizon = float(input("⏳ Simulation Horizon (hours, e.g. 8760): "))
        crews = int(input("👷 Maintenance Crew Size (e.g. 4): "))
        ldt = float(input("📦 Logistical Delay Time (hours, e.g. 2): "))
        strategy = input("🔧 Strategy (corrective/preventive): ").strip().lower()
        pm_int = float(input("📅 PM Interval (hours): ")) if strategy == 'preventive' else 0
        mc_runs = int(input("🔄 Monte Carlo Iterations (e.g. 100): "))

        df = pd.read_csv("config/equipment_data.csv")
        results = []

        print(f"\nCrunching {mc_runs} scenarios...")
        for i in range(mc_runs):
            sim = RAM_V3_Engine(df, horizon, crews, ldt, strategy, pm_int, seed=42+i)
            a, t, m = sim.run()
            results.append({'avail': a, 'tph': t, 'mttr': m})

        res_df = pd.DataFrame(results)
        p50_a = res_df['avail'].median()
        prob_success = len(res_df[res_df['avail'] >= target_a]) / mc_runs

        # FINAL CONSOLIDATED REPORT
        print("\n" + "█"*50)
        print(f"       EXTRACTED RAM PARAMETERS (STRATEGY: {strategy.upper()})")
        print("█"*50)
        print(f"RELIABILITY:   Observed System MTBF: {horizon / res_df['avail'].count():.1f} hrs")
        print(f"AVAILABILITY:  Median: {p50_a:.2%} | Target: {target_a:.2%}")
        print(f"MAINTAINABILITY: Observed Mean Repair Time: {res_df['mttr'].mean():.2f} hrs")
        print("-" * 50)
        print(f"PROBABILITY OF MEETING TARGET: {prob_success:.1%}")
        print(f"CONCLUSION:    {'🟢 ACCEPTABLE RISK' if prob_success > 0.8 else '🔴 HIGH RISK - ADD RESOURCES'}")
        print("█"*50)

        # Advanced Visualization
        fig, ax = plt.subplots(1, 2, figsize=(15, 6))
        sns.histplot(res_df['avail'], kde=True, color='dodgerblue', ax=ax[0])
        ax[0].axvline(target_a, color='red', linestyle='--', label=f'Target ({target_a})')
        ax[0].set_title("Availability Distribution")
        ax[0].legend()

        sns.regplot(x='avail', y='tph', data=res_df, ax=ax[1], color='darkorange')
        ax[1].set_title("Availability vs. Throughput Correlation")
        plt.tight_layout(); plt.show()

    except Exception as e: print(f"Input Error: {e}")

if __name__ == "__main__":
    run_v3_interface()

═══ 💎 INDUSTRY RAM-DES V3.0: PROFESSIONAL DECISION TOOL ═══
🎯 Target System Availability (e.g. 0.85): 0.8
⏳ Simulation Horizon (hours, e.g. 8760): 5000
👷 Maintenance Crew Size (e.g. 4): 3
📦 Logistical Delay Time (hours, e.g. 2): 3
🔧 Strategy (corrective/preventive): corrective
🔄 Monte Carlo Iterations (e.g. 100): 1000

Crunching 1000 scenarios...

██████████████████████████████████████████████████
       EXTRACTED RAM PARAMETERS (STRATEGY: CORRECTIVE)
██████████████████████████████████████████████████
RELIABILITY:   Observed System MTBF: 5.0 hrs
AVAILABILITY:  Median: 98.23% | Target: 80.00%
MAINTAINABILITY: Observed Mean Repair Time: 2930979037.68 hrs
--------------------------------------------------
PROBABILITY OF MEETING TARGET: 79.3%
CONCLUSION:    🔴 HIGH RISK - ADD RESOURCES
██████████████████████████████████████████████████
